## L2 regularization vs Weight Decay

Weight decay, also known as L2 regularization, is a fundamental technique used in deep learning to improve model performance. It acts as a regularizer that penalizes large weights in the network, leading to several benefits:

The weight decay penalty is typically implemented in one of two ways:

- L2 regularization: This directly adds a term to the loss function that is proportional to the sum of the squared weights.
- Weight decay in the optimizer: This modifies the update rule of the optimizer to include a decay factor that reduces the weights at each update step.

<div style="display:flex;">
<img src="./assets/SGD-weight_decay_equals_L2_reg.jpg" width="50%"/>
<img src="./assets/SGD_M_AdamW.jpg" width="50%"/>
</div>

In [2]:
import torch
import torch.nn as nn

When you see lerp_ inside the AdamW source code, it's being used as a mathematically elegant (and computationally fast) way to calculate the Running Exponential Moving Average of the gradients.

1. The Mathematical TranslationThe standard formula for the first moment 
($m_t$) in Adam is:$$m_t = \beta_1 m_{t-1} + (1 - \beta_1)g_t$$

If we rewrite this to match the lerp (Linear Interpolation) formula, where 

$\text{lerp}(start, end, weight) = start + weight \times (end - start)$:

1.Start: exp_avg (the old moving average)
2.End: grad (the new gradient)
3. Weight: 1 - beta1

Substituting these into the lerp formula:

- $\text{out} = exp\_avg + (1 - \beta_1) \times (grad - exp\_avg)$
- $\text{out} = exp\_avg + (1 - \beta_1)grad - exp\_avg + \beta_1 exp\_avg$
- $\text{out} = \beta_1 exp\_avg + (1 - \beta_1)grad$


It is mathematically identical.

2. Why use lerp_ instead of the standard formula?PyTorch developers use lerp_ (the in-place version) for two main reasons:
- Memory Efficiency: lerp_ performs the operation in-place. It doesn't create intermediate tensors for $(1 - \beta)$ or the result of the multiplication, which is crucial when training large models where every byte of VRAM counts.Performance 
- (Fusing): lerp is often implemented as a single "fused" kernel at the CUDA level. Instead of the GPU performing an addition, then a subtraction, then a multiplication, it does the whole interpolation in one pass over the data.

3. Visualizing the UpdateIn AdamW, these two lines handle the "memory" of the optimizer:

| Code Line | Purpose | What it represents |
|:-----|:------:|----:|
| exp_avg.lerp_(grad, 1 - beta1_t) | Momentum | Smooths out the direction of the gradient to avoid oscillations.
| exp_avg_sq.lerp_(grad.square(), 1 - beta2_t) | Volatility | Tracks how much the gradient is "vibrating" to scale the learning rate for each parameter.

In [4]:
x = torch.arange(0, 5, dtype=torch.float32)
grad = torch.arange(0, 5, 1.0)
print(x, grad)

x.lerp_(grad, 1-0.9)
print(x)

tensor([0., 1., 2., 3., 4.]) tensor([0., 1., 2., 3., 4.])
tensor([0., 1., 2., 3., 4.])


In the optim.py file we are using 0-D CPU tensors to avoid recompilation of the compiled graph (need to understand more about torch.compile, dynamo and graph breaks)

https://docs.pytorch.org/docs/stable/user_guide/torch_compiler/compile/programming_model.recompilation.html#wrapping-constants-with-tensors

## Adam

<img src="./assets/Adam.png"/>

https://medium.com/@weidagang/data-compression-with-low-rank-approximation-using-neural-networks-d6a8e8426101

In [6]:
def newton_schulz(G, steps=5, eps=1e-7):
  assert G.ndim == 2
  a, b, c = (3.445, -4.7750, 2.0315)
  X = G.bfloat(16)
  X /= (X.norm() + eps)

  if G.size(0) > G.size(1):
    X = X.T
  
  for _ in range(steps):
    A = X @ X.T
    B = b * A + c * A @ A
    X = a * X + B @ X
  
  if G.size(0) > G.size(1):
    X = X.T
    
  return X

In [7]:
def muon_update(grad, momentum, beta=0.95, ns_steps=5, nesterov=True):
  momentum.lerp_(grad, 1- beta)
  update = grad.lerp_(momentum, beta) if nesterov else momentum

  if update.ndim == 4: # for the case of conv filters [Muon is for 2D]
    update = update.view(len(update), -1)
  
  update = newton_schulz(update, steps=ns_steps)
  update *= max(1, grad.size(-2) / grad.size(-1)) ** 0.5
  return update


In [12]:
x = torch.randn((5, 7), dtype=torch.float16)
x, x[:-1]

(tensor([[ 0.0462, -0.9775,  0.5669,  1.1035, -0.1182, -0.1714,  0.4380],
         [ 1.1279,  0.2057, -0.8813,  1.4844,  1.2637, -0.0274, -1.0254],
         [ 1.8711, -1.5420,  0.6416,  0.6982, -0.8867, -1.8506,  0.0977],
         [ 2.7344, -0.3181,  0.8037, -0.7471,  0.1298,  0.1733, -1.6484],
         [ 0.5161, -0.7837, -1.0791, -0.7168, -0.4885,  0.2065, -0.1156]],
        dtype=torch.float16),
 tensor([[ 0.0462, -0.9775,  0.5669,  1.1035, -0.1182, -0.1714,  0.4380],
         [ 1.1279,  0.2057, -0.8813,  1.4844,  1.2637, -0.0274, -1.0254],
         [ 1.8711, -1.5420,  0.6416,  0.6982, -0.8867, -1.8506,  0.0977],
         [ 2.7344, -0.3181,  0.8037, -0.7471,  0.1298,  0.1733, -1.6484]],
        dtype=torch.float16))